# 星河影音資料庫 EDA

探索 MCP 伺服器的資料庫 `mcp_servers/starriver_db/runtime/starriver.db` 裡有哪些資料，順便找出資料裡埋的「劇情」。

**執行前**：
1. 右上角的 Kernel 選 `cs-agent/.venv`
2. 如果還沒建立資料庫，先在 `cs-agent` 資料夾的終端機執行 `.venv\Scripts\python.exe mcp_servers\starriver_db\init_db.py`

## 1. 連線資料庫

`q()` 是一個小幫手：丟一段 SQL 進去，拿回一張 pandas 表格。

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

DB_PATH = Path("..") / "mcp_servers" / "starriver_db" / "runtime" / "starriver.db"
assert DB_PATH.exists(), "找不到資料庫，請先執行 mcp_servers/starriver_db/init_db.py"

conn = sqlite3.connect(DB_PATH)


def q(sql: str, params: tuple = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params)


q("SELECT * FROM meta")

## 2. 有哪些資料表？各有幾筆？

In [ ]:
tables = q("SELECT name FROM sqlite_master WHERE type = 'table' AND name != 'sqlite_sequence'")["name"]

pd.DataFrame(
    {
        "table": tables,
        "rows": [q(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in tables],
        "columns": [", ".join(q(f"PRAGMA table_info({t})")["name"]) for t in tables],
    }
)

## 3. 每張表的內容

### 3.1 方案 `plans`

In [ ]:
q("SELECT * FROM plans ORDER BY tier")

### 3.2 客戶總覽：`customers` + `subscriptions` + `plans`

三張表用 `customer_id`、`plan_id` 串起來（JOIN），一次看到每位客戶的完整狀態。

In [ ]:
customers = q("""
    SELECT c.customer_id, c.name, s.plan_id, p.max_devices, s.billing_cycle, s.status,
           s.start_date, s.next_billing_date, s.pending_plan, s.paused_until,
           s.pause_count_this_year, s.refunded
    FROM customers c
    JOIN subscriptions s ON s.customer_id = c.customer_id
    JOIN plans p ON p.plan_id = s.plan_id
    ORDER BY c.customer_id
""")
customers

In [ ]:
# 各種狀態、方案、計費週期的分布
for col in ["status", "plan_id", "billing_cycle"]:
    print(customers[col].value_counts().to_string(), "\n")

### 3.3 付款紀錄 `billing_records`

In [ ]:
billing = q("SELECT * FROM billing_records ORDER BY customer_id, billed_at")
billing

In [ ]:
# 每位客戶的付款次數與總金額
billing.groupby("customer_id").agg(payments=("amount", "size"), total=("amount", "sum"))

### 3.4 觀看紀錄 `viewing_history`（使用者行為）

`event` 欄位有兩種：`play` 是正常播放，`blocked_concurrent_limit` 是因為超過同時觀看上限被擋下。

In [ ]:
viewing = q("SELECT * FROM viewing_history ORDER BY customer_id, watched_at")
viewing

In [ ]:
# 每位客戶的觀看行為摘要
plays = viewing[viewing["event"] == "play"]
behavior = pd.DataFrame(
    {
        "sessions": plays.groupby("customer_id").size(),
        "total_minutes": plays.groupby("customer_id")["minutes"].sum(),
        "devices": viewing.groupby("customer_id")["device_name"].nunique(),
        "device_types": viewing.groupby("customer_id")["device_type"].unique().map(", ".join),
        "qualities": plays.groupby("customer_id")["quality"].unique().map(", ".join),
        "blocked": viewing[viewing["event"] == "blocked_concurrent_limit"].groupby("customer_id").size(),
    }
).fillna({"blocked": 0}).astype({"blocked": int})
behavior

### 3.5 歷史客服紀錄 `support_tickets`

In [ ]:
q("SELECT * FROM support_tickets ORDER BY customer_id, created_at")

### 3.6 稽核紀錄 `audit_log`

剛重置的資料庫這裡是空的。等 Agent（或 `try_client.py`）呼叫過工具，回來重跑這格，就能看到 Agent 實際做了哪些事。

In [ ]:
q("SELECT * FROM audit_log ORDER BY log_id")

## 4. 找出資料裡的「劇情」

下面幾個分析，對應到 `evals/tasks.json` 裡的評估題目。

### 4.1 方案跟使用習慣合不合？（對應 T15、T17）

把「實際用了幾台裝置」跟「方案允許幾台同時看」放在一起比較。

In [ ]:
fit = customers[["customer_id", "name", "plan_id", "billing_cycle", "max_devices"]].merge(
    behavior[["devices", "qualities", "blocked"]], left_on="customer_id", right_index=True, how="left"
)


def hint(row) -> str:
    if row["blocked"] > 0:
        return "常被同時觀看上限擋下 → 可能需要升級"
    if row["devices"] == 1 and row["max_devices"] > 1:
        return "只用 1 台裝置 → 方案可能買太大"
    return ""


fit["hint"] = fit.apply(hint, axis=1)
fit

### 4.2 有沒有重複扣款？（對應 T16）

同一位客戶、同一天、同金額出現兩筆以上，就是可疑的重複扣款。

In [ ]:
q("""
    SELECT customer_id, billed_at, amount, COUNT(*) AS times
    FROM billing_records
    GROUP BY customer_id, billed_at, amount
    HAVING COUNT(*) > 1
""")

### 4.3 年繳客戶還在 14 天退款期內嗎？（對應 T10、T11）

開通日當天算第 1 天，所以「第幾天」＝ 系統日期 − 開通日 ＋ 1。

In [ ]:
system_date = pd.Timestamp(q("SELECT value FROM meta WHERE key = 'system_date'")["value"][0])

yearly = customers[customers["billing_cycle"] == "yearly"][["customer_id", "name", "start_date"]].copy()
yearly["day_number"] = (system_date - pd.to_datetime(yearly["start_date"])).dt.days + 1
yearly["refund_eligible"] = yearly["day_number"] <= 14
yearly

### 4.4 安全檢查：自由文字欄位裡有沒有可疑的「指令」？（對應 T12、T16）

`notes`、客服紀錄這類自由文字會被 Agent 讀進上下文，是**提示注入**的入口。

這裡用關鍵字粗略掃描。這只是第一道篩選：攻擊者換個說法就能繞過關鍵字（第二章「輸入清洗」的限制）。

In [ ]:
SUSPICIOUS = ["指令", "忽略", "不需要確認", "直接執行", "免費", "system", "ignore"]

free_text = pd.concat(
    [
        q("SELECT customer_id, 'customers.notes' AS source, notes AS text FROM customers"),
        q("SELECT customer_id, 'support_tickets' AS source, summary || ' / ' || resolution AS text FROM support_tickets"),
    ]
)
free_text["hits"] = free_text["text"].map(lambda t: [w for w in SUSPICIOUS if w.lower() in t.lower()])
free_text[free_text["hits"].map(len) > 0]

## 5. 自己試試看

用 `q()` 寫任何 SQL 查詢。例如：查某位客戶的所有觀看紀錄。

In [ ]:
q("SELECT * FROM viewing_history WHERE customer_id = ? ORDER BY watched_at", ("C004",))

In [ ]:
# 看完記得關閉連線
conn.close()